In [64]:
from __future__ import annotations

import os
import json
from pathlib import Path
from typing import Dict, Any, Tuple

os.environ.setdefault("MPLCONFIGDIR", str(Path(".tmp_mpl").resolve()))

import joblib
import numpy as np
import pandas as pd
import shap


from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve,
    f1_score,
    precision_score,
    recall_score,
)


In [ ]:
# FILE PATHS & BUSINESS ASSUMPTIONS

MODELS_DIR = Path("models").resolve()
ML_RESULTS_JSON = MODELS_DIR / "ml_results.json"
SCORED_PARQUET = Path("data/scored_predictions.parquet").resolve()
XGB_MODEL_PATH = MODELS_DIR / "xgb_model.joblib"
LR_MODEL_PATH = MODELS_DIR / "lr_baseline.joblib"

In [ ]:
# Stated Financial Unit Economics (Used to convert probabilities into Dollars)
DEFAULT_ECONOMICS = {
    "interchange_rate": 0.0175,          # 1.75% earned per approved good transaction
    "false_decline_churn_rate": 0.080,   # 8.0% lost top-of-wallet spend on falsely declined good tx
    "false_decline_friction_usd": 15.00, # $15.00 customer service & abandonment cost per false decline
    "chargeback_fee_usd": 15.00,         # $15.00 operational processing fee per missed fraud
    "fraud_loss_share": 1.00,            # 100% of principal lost on unauthenticated/missed fraud
}

In [ ]:
# Clean business names for dashboard charts

FEATURE_LABELS = {
    "unsecured_ecom": "Unsecured E-Com (No 3DS & No Token)",
    "card_tx_count_1h": "Card Velocity (Tx Count Last 1h)",
    "card_tx_count_24h": "Card Velocity (Tx Count Last 24h)",
    "amount_to_card_avg_ratio": "Spend Ratio vs. Cardholder Avg",
    "log_amount": "Transaction Amount (Log USD)",
    "mcc_train_fraud_rate": "MCC Historical Fraud Rate",
    "is_late_night": "Late-Night Window (01:00–05:00)",
    "is_3ds_authenticated": "3DS Authenticated Indicator",
    "is_tokenized": "Network/Wallet Tokenized",
    "amount_to_mcc_avg_ratio": "Amount Ratio vs. MCC Average",
    "acquirer_train_decline_rate": "Acquirer Historical Decline Rate",
    "is_pre_payday": "Pre-Payday Cash Crunch (Days 15–24)",
    "channel": "Transaction Channel",
    "mcc_risk_tier": "MCC Risk Tier",
}

In [ ]:
# 1. Load the 220,000 authorization transactions

df = pd.read_parquet("../data/authorization_transactions.parquet")
df_3ds = pd.read_parquet("../data/authentication_3ds.parquet")

In [23]:
df



,transaction_id,timestamp,date,month,day_of_month,day_of_week,hour,is_late_night,is_pre_payday,tokenised_card_id,...,card_tx_count_24h,amount_to_card_avg_ratio,is_approved,response_code,decline_reason,decline_type,fraud_flag,has_chargeback,chargeback_reason,chargeback_lifecycle_stage
0,TXN-0000001,2026-07-01 00:01:00,2026-07-01,2026-07,1,2,0,0,0,CRD-005288,...,1,0.539,1,00,Approved,Approved,0,0,None,None
1,TXN-0000002,2026-07-01 00:01:08,2026-07-01,2026-07,1,2,0,0,0,CRD-005381,...,1,0.930,1,00,Approved,Approved,0,0,None,None
2,TXN-0000003,2026-07-01 00:03:32,2026-07-01,2026-07,1,2,0,0,0,CRD-012662,...,1,2.723,1,00,Approved,Approved,0,0,None,None
3,TXN-0000004,2026-07-01 00:03:35,2026-07-01,2026-07,1,2,0,0,0,CRD-015344,...,1,0.444,1,00,Approved,Approved,0,0,None,None
4,TXN-0000005,2026-07-01 00:12:37,2026-07-01,2026-07,1,2,0,0,0,CRD-016434,...,1,0.768,1,00,Approved,Approved,0,0,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
219995,TXN-0219996,2026-10-31 23:52:17,2026-10-31,2026-10,31,5,23,0,0,CRD-000653,...,1,0.100,1,00,Approved,Approved,0,0,None,None
219996,TXN-0219997,2026-10-31 23:53:37,2026-10-31,2026-10,31,5,23,0,0,CRD-002194,...,1,0.117,1,00,Approved,Approved,0,0,None,None
219997,TXN-0219998,2026-10-31 23:55:20,2026-10-31,2026-10,31,5,23,0,0,CRD-016020,...,1,0.118,1,00,Approved,Approved,0,0,None,None
219998,TXN-0219999,2026-10-31 23:56:55,2026-10-31,2026-10,31,5,23,0,0,CRD-003982,...,1,0.179,1,00,Approved,Approved,0,0,None,None


In [24]:
df_3ds

,auth_id,timestamp,date,month,hour,tokenised_card_id,merchant,mcc,mcc_category,mcc_risk_tier,...,channel,amount,ticket_size_bucket,tokenization_indicator,is_tokenized,challenge_mandate,auth_outcome,is_auth_success,failure_reason,latency_ms
0,AUTH-0000001,2026-07-01 00:12:37,2026-07-01,2026-07,0,CRD-016434,Apple Store Online,5732,Consumer Electronics,High,...,E-Commerce,342.34,$100-$500,Tokenized (Network/Wallet),1,Issuer Risk-Based Challenge,Success,1,Authenticated Successfully,1557
1,AUTH-0000002,2026-07-01 00:15:37,2026-07-01,2026-07,0,CRD-002472,Bloomingdales Lux,5944,Luxury Goods & Jewelry,High,...,E-Commerce,502.14,$500+,Raw PAN (Untokenized),0,Issuer Risk-Based Challenge,Success,1,Authenticated Successfully,2160
2,AUTH-0000003,2026-07-01 00:16:47,2026-07-01,2026-07,0,CRD-006784,Zuma Bistro,5812,Restaurants & Dining,Medium,...,E-Commerce,42.29,$25-$100,Raw PAN (Untokenized),0,Frictionless (Low Risk),Success,1,Authenticated Successfully,895
3,AUTH-0000004,2026-07-01 00:16:47,2026-07-01,2026-07,0,CRD-006784,Zuma Bistro,5812,Restaurants & Dining,Medium,...,E-Commerce,42.29,$25-$100,Raw PAN (Untokenized),0,Issuer Risk-Based Challenge,Success,1,Authenticated Successfully,1933
4,AUTH-0000005,2026-07-01 00:16:47,2026-07-01,2026-07,0,CRD-006784,Zuma Bistro,5812,Restaurants & Dining,Medium,...,E-Commerce,42.29,$25-$100,Raw PAN (Untokenized),0,Frictionless (Low Risk),Success,1,Authenticated Successfully,963
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109995,AUTH-0109996,2026-10-31 23:23:11,2026-10-31,2026-10,23,CRD-017717,Etihad Holidays,4511,Airlines & Cross-Border Travel,Medium,...,E-Commerce,120.63,$100-$500,Tokenized (Network/Wallet),1,Frictionless (Low Risk),Success,1,Authenticated Successfully,285
109996,AUTH-0109997,2026-10-31 23:34:07,2026-10-31,2026-10,23,CRD-010568,Lulu Market,5411,Grocery & Supermarkets,Low,...,E-Commerce,34.69,$25-$100,Raw PAN (Untokenized),0,TRA / Low-Value Exemption,Success,1,Authenticated Successfully,601
109997,AUTH-0109998,2026-10-31 23:34:34,2026-10-31,2026-10,23,CRD-013626,NatGrid Power,4900,Utilities & Telecom,Low,...,Recurring / MIT,54.60,$25-$100,Tokenized (Network/Wallet),1,Frictionless (Low Risk),Success,1,Authenticated Successfully,348
109998,AUTH-0109999,2026-10-31 23:39:05,2026-10-31,2026-10,23,CRD-005999,CryptoCom Pay,6051,Crypto & Quasi-Cash,High,...,E-Commerce,176.34,$100-$500,Tokenized (Network/Wallet),1,Frictionless (Low Risk),Success,1,Authenticated Successfully,605


In [25]:
# 2. Create the Time-Based Split masks (Train / Validation / Test)
ts = pd.to_datetime(df["timestamp"])
train_mask = (ts <= "2026-09-15 23:59:59").to_numpy()
val_mask   = ((ts > "2026-09-15 23:59:59") & (ts <= "2026-10-05 23:59:59")).to_numpy()
test_mask  = (ts > "2026-10-05 23:59:59").to_numpy()
print(f"Total rows: {len(df):,}")
print(f"Train rows: {train_mask.sum():,} | Val rows: {val_mask.sum():,} | Test rows: {test_mask.sum():,}")

Total rows: 220,000
Train rows: 130,942 | Val rows: 36,531 | Test rows: 52,527


In [26]:
train_df = df[train_mask]
# Learn historical rates from the Training set only

In [27]:
train_df

,transaction_id,timestamp,date,month,day_of_month,day_of_week,hour,is_late_night,is_pre_payday,tokenised_card_id,...,card_tx_count_24h,amount_to_card_avg_ratio,is_approved,response_code,decline_reason,decline_type,fraud_flag,has_chargeback,chargeback_reason,chargeback_lifecycle_stage
0,TXN-0000001,2026-07-01 00:01:00,2026-07-01,2026-07,1,2,0,0,0,CRD-005288,...,1,0.539,1,00,Approved,Approved,0,0,None,None
1,TXN-0000002,2026-07-01 00:01:08,2026-07-01,2026-07,1,2,0,0,0,CRD-005381,...,1,0.930,1,00,Approved,Approved,0,0,None,None
2,TXN-0000003,2026-07-01 00:03:32,2026-07-01,2026-07,1,2,0,0,0,CRD-012662,...,1,2.723,1,00,Approved,Approved,0,0,None,None
3,TXN-0000004,2026-07-01 00:03:35,2026-07-01,2026-07,1,2,0,0,0,CRD-015344,...,1,0.444,1,00,Approved,Approved,0,0,None,None
4,TXN-0000005,2026-07-01 00:12:37,2026-07-01,2026-07,1,2,0,0,0,CRD-016434,...,1,0.768,1,00,Approved,Approved,0,0,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130937,TXN-0130938,2026-09-15 23:47:38,2026-09-15,2026-09,15,1,23,0,1,CRD-014765,...,1,25.000,0,65,3DS Authentication Required,Soft Decline (Recoverable),0,0,None,None
130938,TXN-0130939,2026-09-15 23:48:30,2026-09-15,2026-09,15,1,23,0,1,CRD-016181,...,1,6.087,1,00,Approved,Approved,0,0,None,None
130939,TXN-0130940,2026-09-15 23:48:39,2026-09-15,2026-09,15,1,23,0,1,CRD-017384,...,2,0.477,1,00,Approved,Approved,0,0,None,None
130940,TXN-0130941,2026-09-15 23:52:51,2026-09-15,2026-09,15,1,23,0,1,CRD-015890,...,1,0.866,1,00,Approved,Approved,0,0,None,None


In [ ]:
mcc_fraud_map = train_df.groupby("mcc")["fraud_flag"].mean().to_dict()
mcc_amt_map   = train_df.groupby("mcc")["amount"].mean().to_dict()
acq_dec_map   = (1.0 - train_df.groupby("acquirer")["is_approved"].mean()).to_dict()

In [36]:
mcc_fraud_map

{'4511': 0.01104187165190773,
 '4900': 0.0020759193357058124,
 '5411': 0.0018037222267770763,
 '5651': 0.003938417472252059,
 '5732': 0.048153034300791556,
 '5812': 0.0031266793688016024,
 '5816': 0.0145413870246085,
 '5912': 0.0018459146993102108,
 '5944': 0.06343612334801763,
 '6051': 0.04957285525207556}

In [38]:
mcc_amt_map

{'4511': 464.2177172843555,
 '4900': 76.98273625939106,
 '5411': 49.331860293514794,
 '5651': 117.95716135577038,
 '5732': 421.69022674802113,
 '5812': 56.76282671356686,
 '5816': 65.37786832853948,
 '5912': 40.997602253959,
 '5944': 731.4382232011748,
 '6051': 433.38781013115147}

In [39]:
acq_dec_map

{'AdyenSwitch': 0.06750961019156843,
 'GlobalPay_Net': 0.06871392791026898,
 'LegacySwitch_Bank': 0.17828477532705556,
 'RegionalPay_ME': 0.08577405857740583,
 'StripeAcq_EU': 0.06564879906268306}

In [45]:
feat = pd.DataFrame(index=df.index)
feat = pd.DataFrame(index=df.index)
feat["log_amount"] = np.log1p(df["amount"])
feat["amount_to_card_avg_ratio"] = df["amount_to_card_avg_ratio"].astype(float)

mcc_avg_amt = df["mcc"].map(mcc_amt_map).fillna(train_df["amount"].mean()).astype(float)
feat["amount_to_mcc_avg_ratio"] = (df["amount"] / mcc_avg_amt.clip(lower=1.0)).clip(0.05, 25.0)

In [46]:
feat

,log_amount,amount_to_card_avg_ratio,amount_to_mcc_avg_ratio
0,5.485834,0.539,0.569731
1,4.390119,0.930,0.171579
2,7.102573,2.723,2.879104
3,4.745888,0.444,2.783334
4,5.838721,0.768,0.811828
...,...,...,...
219995,3.566429,0.100,0.697115
219996,3.538638,0.117,0.588766
219997,3.262318,0.118,0.212874
219998,3.036394,0.179,0.401971


In [44]:
mcc_avg_amt = df["mcc"].map(mcc_amt_map).fillna(train_df["amount"].mean()).astype(float)
feat["amount_to_mcc_avg_ratio"] = (df["amount"] / mcc_avg_amt.clip(lower=1.0)).clip(0.05, 25.0)

In [47]:
#Block 3: Velocity, Security & The unsecured_ecom Interaction Flag

feat["card_tx_count_1h"] = df["card_tx_count_1h"].astype(float)
feat["card_tx_count_24h"] = df["card_tx_count_24h"].astype(float)
feat["is_3ds_authenticated"] = df["is_3ds_authenticated"].astype(float)
feat["is_tokenized"] = df["is_tokenized"].astype(float)
feat["unsecured_ecom"] = (
    (df["channel"] == "E-Commerce")
    & (df["is_3ds_authenticated"] == 0)
    & (df["is_tokenized"] == 0)
).astype(float)

In [48]:
feat

,log_amount,amount_to_card_avg_ratio,amount_to_mcc_avg_ratio,card_tx_count_1h,card_tx_count_24h,is_3ds_authenticated,is_tokenized,unsecured_ecom
0,5.485834,0.539,0.569731,1.0,1.0,1.0,1.0,0.0
1,4.390119,0.930,0.171579,1.0,1.0,0.0,1.0,0.0
2,7.102573,2.723,2.879104,1.0,1.0,1.0,0.0,0.0
3,4.745888,0.444,2.783334,1.0,1.0,0.0,1.0,0.0
4,5.838721,0.768,0.811828,1.0,1.0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...
219995,3.566429,0.100,0.697115,1.0,1.0,0.0,0.0,0.0
219996,3.538638,0.117,0.588766,1.0,1.0,0.0,1.0,0.0
219997,3.262318,0.118,0.212874,1.0,1.0,0.0,0.0,1.0
219998,3.036394,0.179,0.401971,1.0,1.0,0.0,1.0,0.0


In [50]:
# Block 4: Time Flags & Historical Risk Mapping

feat["is_late_night"] = df["is_late_night"].astype(float)
feat["is_pre_payday"] = df["is_pre_payday"].astype(float)
feat["mcc_train_fraud_rate"] = df["mcc"].map(mcc_fraud_map).fillna(train_df["fraud_flag"].mean()).astype(float)
feat["acquirer_train_decline_rate"] = df["acquirer"].map(acq_dec_map).fillna(1.0 - train_df["is_approved"].mean()).astype(float)

In [51]:
feat

,log_amount,amount_to_card_avg_ratio,amount_to_mcc_avg_ratio,card_tx_count_1h,card_tx_count_24h,is_3ds_authenticated,is_tokenized,unsecured_ecom,is_late_night,is_pre_payday,mcc_train_fraud_rate,acquirer_train_decline_rate
0,5.485834,0.539,0.569731,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.048153,0.065649
1,4.390119,0.930,0.171579,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.011042,0.068714
2,7.102573,2.723,2.879104,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.048153,0.085774
3,4.745888,0.444,2.783334,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.001846,0.067510
4,5.838721,0.768,0.811828,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.048153,0.085774
...,...,...,...,...,...,...,...,...,...,...,...,...
219995,3.566429,0.100,0.697115,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.001804,0.178285
219996,3.538638,0.117,0.588766,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.003127,0.067510
219997,3.262318,0.118,0.212874,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.003938,0.067510
219998,3.036394,0.179,0.401971,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.001804,0.085774


In [65]:
train_df


,transaction_id,timestamp,date,month,day_of_month,day_of_week,hour,is_late_night,is_pre_payday,tokenised_card_id,...,card_tx_count_24h,amount_to_card_avg_ratio,is_approved,response_code,decline_reason,decline_type,fraud_flag,has_chargeback,chargeback_reason,chargeback_lifecycle_stage
0,TXN-0000001,2026-07-01 00:01:00,2026-07-01,2026-07,1,2,0,0,0,CRD-005288,...,1,0.539,1,00,Approved,Approved,0,0,None,None
1,TXN-0000002,2026-07-01 00:01:08,2026-07-01,2026-07,1,2,0,0,0,CRD-005381,...,1,0.930,1,00,Approved,Approved,0,0,None,None
2,TXN-0000003,2026-07-01 00:03:32,2026-07-01,2026-07,1,2,0,0,0,CRD-012662,...,1,2.723,1,00,Approved,Approved,0,0,None,None
3,TXN-0000004,2026-07-01 00:03:35,2026-07-01,2026-07,1,2,0,0,0,CRD-015344,...,1,0.444,1,00,Approved,Approved,0,0,None,None
4,TXN-0000005,2026-07-01 00:12:37,2026-07-01,2026-07,1,2,0,0,0,CRD-016434,...,1,0.768,1,00,Approved,Approved,0,0,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130937,TXN-0130938,2026-09-15 23:47:38,2026-09-15,2026-09,15,1,23,0,1,CRD-014765,...,1,25.000,0,65,3DS Authentication Required,Soft Decline (Recoverable),0,0,None,None
130938,TXN-0130939,2026-09-15 23:48:30,2026-09-15,2026-09,15,1,23,0,1,CRD-016181,...,1,6.087,1,00,Approved,Approved,0,0,None,None
130939,TXN-0130940,2026-09-15 23:48:39,2026-09-15,2026-09,15,1,23,0,1,CRD-017384,...,2,0.477,1,00,Approved,Approved,0,0,None,None
130940,TXN-0130941,2026-09-15 23:52:51,2026-09-15,2026-09,15,1,23,0,1,CRD-015890,...,1,0.866,1,00,Approved,Approved,0,0,None,None


In [52]:
dummies = pd.get_dummies(
    df[["channel", "mcc_risk_tier"]].astype(str),
    columns=["channel", "mcc_risk_tier"],
    dtype=float,
)
X_all = pd.concat([feat, dummies], axis=1)
y_all = df["fraud_flag"].to_numpy(dtype=int)

In [53]:
X_all

,log_amount,amount_to_card_avg_ratio,amount_to_mcc_avg_ratio,card_tx_count_1h,card_tx_count_24h,is_3ds_authenticated,is_tokenized,unsecured_ecom,is_late_night,is_pre_payday,mcc_train_fraud_rate,acquirer_train_decline_rate,channel_E-Commerce,channel_POS Chip & PIN,channel_POS Contactless,channel_Recurring / MIT,mcc_risk_tier_High,mcc_risk_tier_Low,mcc_risk_tier_Medium
0,5.485834,0.539,0.569731,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.048153,0.065649,1.0,0.0,0.0,0.0,1.0,0.0,0.0
1,4.390119,0.930,0.171579,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.011042,0.068714,0.0,0.0,1.0,0.0,0.0,0.0,1.0
2,7.102573,2.723,2.879104,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.048153,0.085774,1.0,0.0,0.0,0.0,1.0,0.0,0.0
3,4.745888,0.444,2.783334,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.001846,0.067510,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4,5.838721,0.768,0.811828,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.048153,0.085774,1.0,0.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
219995,3.566429,0.100,0.697115,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.001804,0.178285,0.0,0.0,1.0,0.0,0.0,1.0,0.0
219996,3.538638,0.117,0.588766,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.003127,0.067510,0.0,0.0,1.0,0.0,0.0,0.0,1.0
219997,3.262318,0.118,0.212874,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.003938,0.067510,1.0,0.0,0.0,0.0,0.0,0.0,1.0
219998,3.036394,0.179,0.401971,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.001804,0.085774,0.0,0.0,1.0,0.0,0.0,1.0,0.0


In [54]:
y_all 

array([0, 0, 0, ..., 0, 0, 0])

In [56]:
# 1. Select static columns for the Simple Baseline (Logistic Regression)
from sklearn.pipeline import make_pipeline
baseline_cols = [
    c for c in X_all.columns
    if c in ["log_amount", "is_3ds_authenticated", "is_tokenized", "is_late_night", "is_pre_payday"]
    or c.startswith("channel_")
    or c.startswith("mcc_risk_tier_")
]
# 2. Train Model 1: Simple Baseline (Logistic Regression)
lr_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=500, random_state=42),
)
lr_model.fit(X_all.loc[train_mask, baseline_cols], y_all[train_mask])
# Predict Fraud Probabilities (0.0 to 1.0) using Baseline
lr_prob_val  = lr_model.predict_proba(X_all.loc[val_mask, baseline_cols])[:, 1]
lr_prob_test = lr_model.predict_proba(X_all.loc[test_mask, baseline_cols])[:, 1]
lr_prob_all  = lr_model.predict_proba(X_all[baseline_cols])[:, 1]

In [58]:
# 3. Train Model 2: Advanced Model (XGBoost with Velocity & Spend Ratio Features)
xgb_model = XGBClassifier(
    n_estimators=240,
    max_depth=6,
    learning_rate=0.055,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=12,
    reg_lambda=2.0,
    eval_metric="logloss",
    random_state=42,
    n_jobs=4,
)
xgb_model.fit(
    X_all.loc[train_mask],
    y_all[train_mask],
    eval_set=[(X_all.loc[val_mask], y_all[val_mask])],
    verbose=False,
)

# Predict Fraud Probabilities (0.0 to 1.0) using XGBoost
xgb_prob_val  = xgb_model.predict_proba(X_all.loc[val_mask])[:, 1]
xgb_prob_test = xgb_model.predict_proba(X_all.loc[test_mask])[:, 1]
xgb_prob_all  = xgb_model.predict_proba(X_all)[:, 1]

# 4. Compare Out-of-Time Test Set Metrics
y_test = y_all[test_mask]
print("=== OUT-OF-TIME TEST SET RESULTS (Oct 6 – Oct 31, 2026) ===")
print(f"Logistic Regression Baseline -> ROC-AUC: {roc_auc_score(y_test, lr_prob_test):.4f} | PR-AUC: {average_precision_score(y_test, lr_prob_test):.4f}")
print(f"XGBoost Advanced Model       -> ROC-AUC: {roc_auc_score(y_test, xgb_prob_test):.4f} | PR-AUC: {average_precision_score(y_test, xgb_prob_test):.4f}")

=== OUT-OF-TIME TEST SET RESULTS (Oct 6 – Oct 31, 2026) ===
Logistic Regression Baseline -> ROC-AUC: 0.9145 | PR-AUC: 0.5135
XGBoost Advanced Model       -> ROC-AUC: 0.9373 | PR-AUC: 0.6161


In [59]:
# Stated Financial Unit Economics (Bank Assumptions)
DEFAULT_ECONOMICS = {
    "interchange_rate": 0.0175,          # +1.75% fee earned by bank on approved good spend
    "false_decline_churn_rate": 0.080,   # -8.0% lost future card spend when a good customer is blocked
    "false_decline_friction_usd": 15.00, # -$15.00 support/friction cost per false decline
    "chargeback_fee_usd": 15.00,         # -$15.00 operational dispute fee per missed fraud
    "fraud_loss_share": 1.00,            # -100% of transaction amount lost when fraud is missed
}

def compute_threshold_economics(y_true, y_prob, amounts, thresholds, econ=DEFAULT_ECONOMICS):
    ic_rate = econ["interchange_rate"]
    churn_rate = econ["false_decline_churn_rate"]
    fd_fee = econ["false_decline_friction_usd"]
    cb_fee = econ["chargeback_fee_usd"]
    f_share = econ["fraud_loss_share"]

    is_fraud = (y_true == 1)
    is_good  = ~is_fraud
    total_fraud_usd = float(amounts[is_fraud].sum())

    rows = []
    for tau in thresholds:
        flagged  = (y_prob >= tau)  # Model says: Block / Decline
        approved = ~flagged         # Model says: Approve

        tp = flagged & is_fraud   # Caught Fraud (Money Saved!)
        fp = flagged & is_good    # False Decline (Good customer blocked -> Money Lost!)
        fn = approved & is_fraud  # Missed Fraud
        tn = approved & is_good   # Approved Good Customer

        tp_cnt, fp_cnt, fn_cnt, tn_cnt = int(tp.sum()), int(fp.sum()), int(fn.sum()), int(tn.sum())

        # Dollar calculations
        interchange_earned_usd = float(amounts[tn].sum() * ic_rate)
        fraud_prevented_usd    = float(amounts[tp].sum() * f_share + tp_cnt * cb_fee)
        missed_fraud_loss_usd  = float(amounts[fn].sum() * f_share + fn_cnt * cb_fee)
        false_decline_cost_usd = float(amounts[fp].sum() * (ic_rate + churn_rate) + fp_cnt * fd_fee)

        net_ml_savings_usd = fraud_prevented_usd - false_decline_cost_usd
        net_portfolio_profit_usd = interchange_earned_usd - missed_fraud_loss_usd - false_decline_cost_usd

        precision = tp_cnt / max(tp_cnt + fp_cnt, 1)
        recall    = tp_cnt / max(int(is_fraud.sum()), 1)
        f1        = 2 * precision * recall / max(precision + recall, 1e-9)

        rows.append({
            "threshold": round(float(tau), 2),
            "approval_rate": round(int(approved.sum()) / max(len(y_true), 1), 4),
            "precision": round(precision, 4),
            "recall_tx": round(recall, 4),
            "recall_usd": round(float(amounts[tp].sum()) / max(total_fraud_usd, 1.0), 4),
            "f1_score": round(f1, 4),
            "false_positive_rate": round(fp_cnt / max(int(is_good.sum()), 1), 4),
            "tp_caught_fraud": tp_cnt,
            "fp_false_declines": fp_cnt,
            "fn_missed_fraud": fn_cnt,
            "tn_approved_good": tn_cnt,
            "interchange_earned_usd": round(interchange_earned_usd, 2),
            "fraud_prevented_usd": round(fraud_prevented_usd, 2),
            "missed_fraud_loss_usd": round(missed_fraud_loss_usd, 2),
            "false_decline_cost_usd": round(false_decline_cost_usd, 2),
            "net_ml_savings_usd": round(net_ml_savings_usd, 2),
            "net_portfolio_profit_usd": round(net_portfolio_profit_usd, 2),
        })
    return pd.DataFrame(rows)

# 1. Test thresholds from 0.06 to 0.90 on the Validation Set to find the best cutoff (tau*)
amt_all = df["amount"].to_numpy(dtype=float)
thresholds = np.round(np.arange(0.06, 0.92, 0.02), 2)

val_econ_xgb = compute_threshold_economics(y_all[val_mask], xgb_prob_val, amt_all[val_mask], thresholds)
optimal_threshold = float(val_econ_xgb.loc[val_econ_xgb["net_ml_savings_usd"].idxmax(), "threshold"])

val_econ_lr = compute_threshold_economics(y_all[val_mask], lr_prob_val, amt_all[val_mask], thresholds)
optimal_lr_threshold = float(val_econ_lr.loc[val_econ_lr["net_ml_savings_usd"].idxmax(), "threshold"])

# 2. Evaluate on the unseen Out-of-Time Test Set (Oct 6 - Oct 31)
test_econ_xgb = compute_threshold_economics(y_all[test_mask], xgb_prob_test, amt_all[test_mask], thresholds)
test_econ_lr  = compute_threshold_economics(y_all[test_mask], lr_prob_test, amt_all[test_mask], thresholds)

xgb_at_opt = test_econ_xgb[test_econ_xgb["threshold"] == optimal_threshold].iloc[0]
lr_at_opt  = test_econ_lr[test_econ_lr["threshold"] == optimal_lr_threshold].iloc[0]

# 3. Compare against Legacy Static Rules ('59 Suspected Fraud' & '05 Do Not Honor')
legacy_flagged_test = df.loc[test_mask, "response_code"].isin(["59", "05"]).to_numpy(dtype=float)
legacy_at_test = compute_threshold_economics(y_all[test_mask], legacy_flagged_test, amt_all[test_mask], np.array([0.50])).iloc[0]

print(f"Optimal Operating Threshold (chosen on Validation Set): tau* = {optimal_threshold:.2f}\n")
print("=== BUSINESS IMPACT ON OUT-OF-TIME TEST SET (26 Days) ===")
print(f"1. Legacy Static Rules  -> Net Savings: ${legacy_at_test['net_ml_savings_usd']:,.2f} | Fraud Caught: {legacy_at_test['recall_tx']:.1%} | False Declines: {int(legacy_at_test['fp_false_declines']):,}")
print(f"2. LogReg Baseline      -> Net Savings: ${lr_at_opt['net_ml_savings_usd']:,.2f} | Fraud Caught: {lr_at_opt['recall_tx']:.1%} | False Declines: {int(lr_at_opt['fp_false_declines']):,}")
print(f"3. XGBoost (@ tau=0.12) -> Net Savings: ${xgb_at_opt['net_ml_savings_usd']:,.2f} | Fraud Caught: {xgb_at_opt['recall_tx']:.1%} | False Declines: {int(xgb_at_opt['fp_false_declines']):,}")
print(f"\nIncremental Dollar Gain (XGBoost vs. Legacy Rules): +${xgb_at_opt['net_ml_savings_usd'] - legacy_at_test['net_ml_savings_usd']:,.2f}")

Optimal Operating Threshold (chosen on Validation Set): tau* = 0.12

=== BUSINESS IMPACT ON OUT-OF-TIME TEST SET (26 Days) ===
1. Legacy Static Rules  -> Net Savings: $456,388.37 | Fraud Caught: 69.2% | False Declines: 1,102
2. LogReg Baseline      -> Net Savings: $546,108.99 | Fraud Caught: 63.4% | False Declines: 588
3. XGBoost (@ tau=0.12) -> Net Savings: $573,910.15 | Fraud Caught: 69.5% | False Declines: 615

Incremental Dollar Gain (XGBoost vs. Legacy Rules): +$117,521.78


In [60]:
import shap

# Clean business names for each feature column
FEATURE_LABELS = {
    "unsecured_ecom": "Unsecured E-Com (No 3DS & No Token)",
    "card_tx_count_1h": "Card Velocity (Tx Count Last 1h)",
    "card_tx_count_24h": "Card Velocity (Tx Count Last 24h)",
    "amount_to_card_avg_ratio": "Spend Ratio vs. Cardholder Avg",
    "log_amount": "Transaction Amount (Log USD)",
    "mcc_train_fraud_rate": "MCC Historical Fraud Rate",
    "is_late_night": "Late-Night Window (01:00–05:00)",
    "is_3ds_authenticated": "3DS Authenticated Indicator",
    "is_tokenized": "Network/Wallet Tokenized",
    "amount_to_mcc_avg_ratio": "Amount Ratio vs. MCC Average",
    "acquirer_train_decline_rate": "Acquirer Historical Decline Rate",
    "is_pre_payday": "Pre-Payday Cash Crunch (Days 15–24)",
    "channel_E-Commerce": "Channel: E-Commerce",
    "channel_POS Chip & PIN": "Channel: POS Chip & PIN",
    "channel_POS Contactless": "Channel: POS Contactless",
    "channel_Recurring / MIT": "Channel: Recurring / MIT",
    "mcc_risk_tier_High": "MCC Risk Tier: High",
    "mcc_risk_tier_Medium": "MCC Risk Tier: Medium",
    "mcc_risk_tier_Low": "MCC Risk Tier: Low",
}

# 1. Global SHAP Explainability (Across 3,000 Test Transactions)
explainer = shap.TreeExplainer(xgb_model)
X_test_df = X_all.loc[test_mask]
shap_sample = X_test_df.sample(n=min(3000, len(X_test_df)), random_state=42)
shap_values = explainer.shap_values(shap_sample)

# Calculate average absolute impact of each feature: mean(|SHAP|)
mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_summary_df = pd.DataFrame({
    "Feature Code": X_all.columns,
    "Business Feature Name": [FEATURE_LABELS.get(c, c) for c in X_all.columns],
    "Mean |SHAP| Impact": np.round(mean_abs_shap, 4),
}).sort_values("Mean |SHAP| Impact", ascending=False).reset_index(drop=True)

print("=== TOP 10 GLOBAL SHAP FEATURE DRIVERS ===")
display(shap_summary_df.head(10))

# 2. Risk Segments Breakdown on Out-of-Time Test Set
test_df = df.loc[test_mask].copy()
test_df["ml_fraud_prob_xgb"] = xgb_prob_test
test_df["ml_risk_segment"] = np.where(
    xgb_prob_test >= optimal_threshold,
    "3. High Risk (Auto-Block >= 0.12)",
    np.where(xgb_prob_test >= 0.08, "2. Medium Risk (3DS Challenge 0.08-0.12)", "1. Low Risk (Frictionless Approve < 0.08)"),
)

seg_table = (
    test_df.groupby("ml_risk_segment")
    .agg(
        Transactions=("transaction_id", "count"),
        Avg_Predicted_Risk_Pct=("ml_fraud_prob_xgb", lambda s: round(s.mean() * 100, 2)),
        Actual_Fraud_Rate_Pct=("fraud_flag", lambda s: round(s.mean() * 100, 2)),
        Fraud_Caught_Count=("fraud_flag", "sum"),
    )
    .reset_index()
)
seg_table["Share_of_Volume_Pct"] = (seg_table["Transactions"] / len(test_df) * 100).round(2)
seg_table["Share_of_Total_Fraud_Pct"] = (seg_table["Fraud_Caught_Count"] / test_df["fraud_flag"].sum() * 100).round(2)

print("\n=== RISK SEGMENTS TABLE (OUT-OF-TIME TEST SET) ===")
display(seg_table)

=== TOP 10 GLOBAL SHAP FEATURE DRIVERS ===


,Feature Code,Business Feature Name,Mean |SHAP| Impact
0,unsecured_ecom,Unsecured E-Com (No 3DS & No Token),0.6099
1,amount_to_card_avg_ratio,Spend Ratio vs. Cardholder Avg,0.4786
2,mcc_train_fraud_rate,MCC Historical Fraud Rate,0.4287
3,is_3ds_authenticated,3DS Authenticated Indicator,0.2999
4,is_tokenized,Network/Wallet Tokenized,0.2880
5,log_amount,Transaction Amount (Log USD),0.1925
6,card_tx_count_1h,Card Velocity (Tx Count Last 1h),0.1240
7,acquirer_train_decline_rate,Acquirer Historical Decline Rate,0.1176
8,is_late_night,Late-Night Window (01:00–05:00),0.1104
9,amount_to_mcc_avg_ratio,Amount Ratio vs. MCC Average,0.0950



=== RISK SEGMENTS TABLE (OUT-OF-TIME TEST SET) ===


,ml_risk_segment,Transactions,Avg_Predicted_Risk_Pct,Actual_Fraud_Rate_Pct,Fraud_Caught_Count,Share_of_Volume_Pct,Share_of_Total_Fraud_Pct
0,1. Low Risk (Frictionless Approve < 0.08),51014,0.38,0.38,194,97.12,24.31
1,2. Medium Risk (3DS Challenge 0.08-0.12),343,9.79,14.29,49,0.65,6.14
2,3. High Risk (Auto-Block >= 0.12),1170,47.68,47.44,555,2.23,69.55


In [63]:
import joblib

# 1. Local SHAP Explainability: Inspect Why the #1 Highest-Risk Test Transaction Was Flagged
top_test_indices = X_test_df.iloc[np.argsort(-xgb_prob_test)[:50]].index
shap_top50 = explainer.shap_values(X_all.loc[top_test_indices])
base_value = float(np.atleast_1d(explainer.expected_value)[0])

sample_idx = top_test_indices[0]
sample_tx  = df.loc[sample_idx]
sample_shap_df = pd.DataFrame({
    "Feature": [FEATURE_LABELS.get(c, c) for c in X_all.columns],
    "Transaction Value": [round(float(X_all.loc[sample_idx, c]), 3) for c in X_all.columns],
    "SHAP Contribution (Log-Odds)": np.round(shap_top50[0], 4),
}).sort_values("SHAP Contribution (Log-Odds)", key=abs, ascending=False)

print(f"=== LOCAL SHAP EXPLANATION FOR TRANSACTION {sample_tx['transaction_id']} ===")
print(f"Merchant: {sample_tx['merchant']} ({sample_tx['mcc_category']}) | Amount: ${sample_tx['amount']:,.2f} | Channel: {sample_tx['channel']}")
print(f"Predicted Fraud Probability: {xgb_prob_all[sample_idx]:.2%} | Actual Fraud Flag: {sample_tx['fraud_flag']}")
display(sample_shap_df.head(8))

# 2. Save Trained Models & Scored Predictions to ../models and ../data
MODELS_DIR = Path("../models").resolve()
DATA_DIR   = Path("../data").resolve()
MODELS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

LR_MODEL_PATH  = MODELS_DIR / "lr_baseline.joblib"
XGB_MODEL_PATH = MODELS_DIR / "xgb_model.joblib"
SCORED_PARQUET = DATA_DIR / "scored_predictions.parquet"

joblib.dump(lr_model, LR_MODEL_PATH)
joblib.dump(xgb_model, XGB_MODEL_PATH)

risk_segment_all = np.where(
    xgb_prob_all >= optimal_threshold,
    "High Risk (Auto-Block / Step-Up)",
    np.where(xgb_prob_all >= 0.08, "Medium Risk (3DS Challenge)", "Low Risk (Frictionless Approve)"),
)
split_name = np.where(train_mask, "1_Train", np.where(val_mask, "2_Validation", "3_Test_OOT"))

scored_df = pd.DataFrame({
    "transaction_id": df["transaction_id"],
    "split": pd.Series(split_name, dtype="string"),
    "ml_fraud_prob_xgb": np.round(xgb_prob_all.astype(np.float32), 4),
    "ml_fraud_prob_lr": np.round(lr_prob_all.astype(np.float32), 4),
    "ml_risk_segment": pd.Series(risk_segment_all, dtype="string"),
    "ml_top_drivers": pd.Series(np.where(xgb_prob_all >= optimal_threshold, "High Risk Signal", "Baseline Normal Profile"), dtype="string"),
})
scored_df.to_parquet(SCORED_PARQUET, index=False)

print(f"\nSaved trained models to: {MODELS_DIR}")
print(f"Saved {len(scored_df):,} scored predictions to: {SCORED_PARQUET}")

=== LOCAL SHAP EXPLANATION FOR TRANSACTION TXN-0172431 ===
Merchant: Virgin Megastore (Consumer Electronics) | Amount: $1,390.71 | Channel: E-Commerce
Predicted Fraud Probability: 99.45% | Actual Fraud Flag: 1


,Feature,Transaction Value,SHAP Contribution (Log-Odds)
7,Unsecured E-Com (No 3DS & No Token),1.000,2.4768
1,Spend Ratio vs. Cardholder Avg,12.026,1.7063
3,Card Velocity (Tx Count Last 1h),4.000,1.5090
8,Late-Night Window (01:00–05:00),1.000,1.3232
0,Transaction Amount (Log USD),7.238,0.8068
10,MCC Historical Fraud Rate,0.048,0.5888
5,3DS Authenticated Indicator,0.000,0.4156
4,Card Velocity (Tx Count Last 24h),4.000,0.3845



Saved trained models to: C:\Users\DELL\Desktop\NI\models
Saved 220,000 scored predictions to: C:\Users\DELL\Desktop\NI\data\scored_predictions.parquet
